# Feature Importance and SHAP

"Which features matter?" has at least three different answers, and they disagree
often enough that the disagreement is the interesting part.

| method | question it answers | cost | trustworthy? |
|---|---|---|---|
| **impurity / gain** | how much did this feature reduce the loss while *training*? | free | **biased** — see §1 |
| **permutation** | how much worse does the model get if I destroy this feature? | $O(p)$ refits of prediction | yes, with care |
| **SHAP** | for *this* prediction, how much did each feature contribute? | $O(2^p)$ exactly | yes, and it is the only one that is per-prediction |

The first is the one printed by `.feature_importances_` on every tree model, the
one people quote, and the one with a known systematic flaw. This page builds all
three by hand and shows exactly where the first one lies.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True, linewidth=140)


def banner(title, ch="="):
    print(ch * 78)
    print(f"  {title}")
    print(ch * 78)


def step(label):
    print()
    print(label)
    print("=" * 78)


def check(manual, from_array, label="MATCH"):
    ok = np.allclose(manual, from_array)
    print(f"      = {manual:+.6f}   |  from array: {from_array:+.6f}   |  {label}: {ok}")
    return ok


def expand(u, v, per_line=4):
    """Write a dot product out term by term."""
    terms = [f"({a:+.4f})*({b:+.4f})" for a, b in zip(u, v)]
    lines, out = [], ""
    for i, t in enumerate(terms):
        out += t + (" + " if i < len(terms) - 1 else "")
        if (i + 1) % per_line == 0:
            lines.append(out); out = ""
    if out:
        lines.append(out)
    return "\n      ".join(lines)


def show_matrix(M, name="M", rows=None, cols=None, fmt="{:+8.3f}", max_cols=14):
    M = np.asarray(M)
    if M.ndim == 1:
        M = M[None, :]
    cut = M.shape[1] > max_cols
    Ms = M[:, :max_cols] if cut else M
    print(f"{name} {np.asarray(M).shape}" +
          (f"   [first {max_cols} of {M.shape[1]} columns]" if cut else "") + " =")
    if cols is not None:
        head = "".join(f"{str(c):>10}" for c in (cols[:max_cols] if cut else cols))
        print(" " * 13 + head)
    for r in range(Ms.shape[0]):
        lab = f"{str(rows[r]):>10} | " if rows is not None else f"{r:>10} | "
        print("   " + lab + "  ".join(fmt.format(v) for v in Ms[r]))


def show_matmul(A, B, C, na="A", nb="B", nc="C", i=0, j=0,
                rows_A=None, note=None, max_cols=14):
    """Print both operands of A @ B = C, then verify one element by hand."""
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{na} {A.shape}   @   {nb} {B.shape}   =   {nc} {C.shape}")
    print(f"  shape rule: ({A.shape[0]} x {A.shape[1]}) @ ({B.shape[0]} x {B.shape[1]})"
          f" -> ({A.shape[0]} x {B.shape[1]})   inner dim {A.shape[1]} cancels")
    if note:
        print(f"  {note}")
    print("=" * 78)
    print()
    show_matrix(A, na, rows_A, max_cols=max_cols)
    print()
    show_matrix(B, nb, max_cols=max_cols)
    print()
    show_matrix(C, nc, rows_A, max_cols=max_cols)
    print(f"\n  worked element  {nc}[{i},{j}]  =  row {i} of {na}  .  column {j} of {nb}")
    print(f"      row {i} of {na} = {np.round(A[i], 4)}")
    print(f"      col {j} of {nb} = {np.round(B[:, j], 4)}")
    print(f"      {expand(A[i], B[:, j])}")
    check(float(A[i] @ B[:, j]), float(C[i, j]))


def show_elementwise(A, B, C, na, nb, nc, op="+", rows=None):
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{nc} = {na} {op} {nb}      all three are {A.shape} - element-wise")
    for r in range(A.shape[0]):
        lab = f"{rows[r]:>10}" if rows is not None else f"{r:>10}"
        print(f"\n{lab} |")
        print("   " + f"{na:>14}: " + "  ".join(f"{v:+8.3f}" for v in A[r]))
        print("   " + f"{nb:>14}: " + "  ".join(f"{v:+8.3f}" for v in B[r]))
        print("   " + f"{nc:>14}: " + "  ".join(f"{v:+8.3f}" for v in C[r]))
    print(f"\n  check [0,0]: {A[0,0]:+.6f} {op} {B[0,0]:+.6f} = {C[0,0]:+.6f}   "
          f"MATCH: {np.isclose(A[0,0] + B[0,0] if op == '+' else A[0,0] - B[0,0], C[0,0])}")


print("Display helpers ready: banner, step, check, expand, show_matrix, "
      "show_matmul, show_elementwise.")

# ----------------------------------------------------------------------------
#  A deliberately tiny EEG-flavoured table: 12 recordings, hand-checkable.
# ----------------------------------------------------------------------------
feature_names = ["line_length", "spike_rate", "age"]
region_names  = ["frontal", "temporal", "occipital"]

X = np.array([
    [12.0,  0.5, 24.0],
    [15.0,  1.2, 31.0],
    [42.0,  6.8, 58.0],
    [38.0,  5.1, 47.0],
    [11.0,  0.3, 19.0],
    [55.0,  8.2, 63.0],
    [40.0,  1.9, 35.0],
    [47.0,  7.4, 52.0],
    [14.0,  0.8, 28.0],
    [51.0,  6.1, 60.0],
    [45.0,  2.4, 41.0],
    [35.0,  4.7, 30.0],
])

# categorical column, used by the CatBoost notebook
region = np.array([0, 1, 1, 2, 0, 1, 2, 1, 0, 2, 0, 1])

# binary target: did a seizure follow?
y_class = np.array([0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1])

# continuous target: seizure burden in minutes/hour
y_reg = np.array([0.2, 0.5, 6.4, 4.8, 0.1, 9.1, 3.5, 7.2, 0.3, 6.9, 1.6, 4.1])

n_samples, n_features = X.shape
rec_ids = [f"rec{i:02d}" for i in range(n_samples)]

print(f"{n_samples} recordings x {n_features} features\n")
hdr = f"{'id':>6}  {'region':>10}  " + "  ".join(f"{f:>12}" for f in feature_names)
hdr += f"  {'y_class':>8}  {'y_reg':>7}"
print(hdr)
print("-" * len(hdr))
for i in range(n_samples):
    print(f"{rec_ids[i]:>6}  {region_names[region[i]]:>10}  " +
          "  ".join(f"{v:>12.1f}" for v in X[i]) +
          f"  {y_class[i]:>8}  {y_reg[i]:>7.1f}")
print("-" * len(hdr))
print(f"\nclass balance: {int((y_class==0).sum())} negative, {int((y_class==1).sum())} positive")
print(f"y_reg mean = {y_reg.mean():.4f}, std = {y_reg.std():.4f}")

Display helpers ready: banner, step, check, expand, show_matrix, show_matmul, show_elementwise.
12 recordings x 3 features

    id      region   line_length    spike_rate           age   y_class    y_reg
-------------------------------------------------------------------------------
 rec00     frontal          12.0           0.5          24.0         0      0.2
 rec01    temporal          15.0           1.2          31.0         0      0.5
 rec02    temporal          42.0           6.8          58.0         1      6.4
 rec03   occipital          38.0           5.1          47.0         1      4.8
 rec04     frontal          11.0           0.3          19.0         0      0.1
 rec05    temporal          55.0           8.2          63.0         1      9.1
 rec06   occipital          40.0           1.9          35.0         1      3.5
 rec07    temporal          47.0           7.4          52.0         1      7.2
 rec08     frontal          14.0           0.8          28.0         0      

## 1. Impurity importance and its cardinality bias

A tree's built-in importance for a feature is the total loss reduction across
every split that used it, weighted by how many samples reached each node.

The flaw: a feature with **many distinct values** offers many candidate split
points. More candidates means a better chance that one of them fits the noise in
the training data. So high-cardinality features accumulate impurity reduction
*even when they carry no signal at all*.

The demonstration is the cleanest possible: add a column of pure random noise
with a distinct value for every row, and watch it climb the importance ranking.

In [2]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

banner("IMPURITY IMPORTANCE IS FOOLED BY CARDINALITY")

rng = np.random.RandomState(0)
n_big = 800
Xr = np.column_stack([
    rng.randn(n_big),                       # real signal
    rng.randint(0, 2,  n_big),              # binary,  useless
    rng.randint(0, 10, n_big),              # 10 values, useless
    rng.rand(n_big),                        # continuous, useless, ALL distinct
])
names = ['signal', 'noise_binary', 'noise_10vals', 'noise_continuous']
yr = (Xr[:, 0] + rng.randn(n_big)*0.5 > 0).astype(int)

print(f"  {n_big} rows, 4 features. Only 'signal' has any relationship to y.")
print(f"  {'feature':>18}  {'distinct values':>16}")
for j, nm in enumerate(names):
    print(f"  {nm:>18}  {len(np.unique(Xr[:, j])):>16}")

step("What the model's built-in importance says")
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xr, yr)
imp = rf.feature_importances_
order = np.argsort(-imp)
print(f"  {'rank':>5}  {'feature':>18}  {'impurity importance':>20}")
print("  " + "-" * 48)
for r, j in enumerate(order, 1):
    flag = "  <- pure noise" if j != 0 else ""
    print(f"  {r:>5}  {names[j]:>18}  {imp[j]:>20.4f}{flag}")
print("  " + "-" * 48)
print(f"  The three noise features share"
      f" {imp[1:].sum():.1%} of the total importance.")
print(f"  And they are ranked by CARDINALITY, not by usefulness:")
print(f"    binary {imp[1]:.4f}  <  10 values {imp[2]:.4f}"
      f"  <  all-distinct {imp[3]:.4f}")
print(f"  ordering follows cardinality:"
      f" {imp[1] < imp[2] < imp[3]}")

step("What permutation importance says")
Xtr, Xte, ytr, yte = train_test_split(Xr, yr, test_size=0.4, random_state=0)
rf2 = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xtr, ytr)
pi = permutation_importance(rf2, Xte, yte, n_repeats=30, random_state=0)
print(f"  {'feature':>18}  {'drop in accuracy':>17}  {'std':>7}")
print("  " + "-" * 46)
for j in np.argsort(-pi.importances_mean):
    print(f"  {names[j]:>18}  {pi.importances_mean[j]:>17.4f}"
          f"  {pi.importances_std[j]:>7.4f}")
print("  " + "-" * 46)
print(f"  The noise features score ~0, as they should: destroying a column")
print(f"  that carries nothing costs nothing.")
print()
print(f"  Two reasons permutation is honest here. It is measured on HELD-OUT")
print(f"  data, so fitting noise does not help. And it asks what the model")
print(f"  actually USES, not what it looked at during training.")

  IMPURITY IMPORTANCE IS FOOLED BY CARDINALITY
  800 rows, 4 features. Only 'signal' has any relationship to y.
             feature   distinct values
              signal               800
        noise_binary                 2
        noise_10vals                10
    noise_continuous               800

What the model's built-in importance says
   rank             feature   impurity importance
  ------------------------------------------------
      1              signal                0.7434
      2    noise_continuous                0.1688  <- pure noise
      3        noise_10vals                0.0698  <- pure noise
      4        noise_binary                0.0180  <- pure noise
  ------------------------------------------------
  The three noise features share 25.7% of the total importance.
  And they are ranked by CARDINALITY, not by usefulness:
    binary 0.0180  <  10 values 0.0698  <  all-distinct 0.1688
  ordering follows cardinality: True

What permutation importance say

### The catch with permutation importance

It is not a free lunch. When two features are **correlated**, permuting one
leaves the information intact in the other, so *both* look unimportant — the
model just routes around the damage. The result is that genuinely important but
redundant features get scored near zero, which reads as "irrelevant" and is not.

Permuting also creates rows that could never occur — a 2-metre-tall person
weighing 30kg — and asks the model to predict on them. The number you get is
partly a measurement of the model's behaviour off the data manifold.

In [3]:
banner("PERMUTATION IMPORTANCE AND CORRELATED FEATURES")

step("Duplicate the signal column and re-measure")
Xd = np.column_stack([Xr[:, 0], Xr[:, 0] + rng.randn(n_big)*0.01, Xr[:, 1:]])
names_d = ['signal', 'signal_copy'] + names[1:]
Xtr, Xte, ytr, yte = train_test_split(Xd, yr, test_size=0.4, random_state=0)
rf3 = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xtr, ytr)
pi3 = permutation_importance(rf3, Xte, yte, n_repeats=30, random_state=0)
print(f"  correlation between the two signal columns:"
      f" {np.corrcoef(Xd[:,0], Xd[:,1])[0,1]:.4f}")
print()
print(f"  {'feature':>18}  {'permutation importance':>23}")
print("  " + "-" * 44)
for j in np.argsort(-pi3.importances_mean):
    print(f"  {names_d[j]:>18}  {pi3.importances_mean[j]:>23.4f}")
print("  " + "-" * 44)
print(f"  Both signal columns now score far below what 'signal' scored alone")
print(f"  ({pi.importances_mean[0]:.4f}). Neither is unimportant -- between them")
print(f"  they carry the entire model. Permuting one leaves the other intact,")
print(f"  so the model barely notices, and both look dispensable.")
print()
print(f"  The correct reading is 'this feature is not UNIQUELY important',")
print(f"  which is a different statement from 'this feature does not matter'.")

  PERMUTATION IMPORTANCE AND CORRELATED FEATURES

Duplicate the signal column and re-measure
  correlation between the two signal columns: 1.0000

             feature   permutation importance
  --------------------------------------------
         signal_copy                   0.1615
              signal                   0.1315
        noise_10vals                   0.0067
    noise_continuous                   0.0035
        noise_binary                   0.0004
  --------------------------------------------
  Both signal columns now score far below what 'signal' scored alone
  (0.3396). Neither is unimportant -- between them
  they carry the entire model. Permuting one leaves the other intact,
  so the model barely notices, and both look dispensable.

  The correct reading is 'this feature is not UNIQUELY important',
  which is a different statement from 'this feature does not matter'.


## 2. SHAP: Shapley values, from the actual definition

SHAP borrows a result from cooperative game theory. Treat a prediction as a
payout and the features as players who cooperated to produce it. The Shapley
value is the **unique** way to divide that payout satisfying a short list of
fairness axioms.

For feature $j$ at a particular row:

$$\phi_j = \sum_{S \subseteq F \setminus \{j\}} \frac{|S|!\,(|F| - |S| - 1)!}{|F|!}\Big[ v(S \cup \{j\}) - v(S) \Big]$$

Read it as: average the **marginal contribution** of $j$ — what it adds to a
coalition $S$ that does not already contain it — over every possible coalition,
weighted so that every *ordering* of the features counts equally.

The weight is less mysterious than it looks. There are $|F|!$ orderings of the
features. $|S|!$ of them put exactly the members of $S$ before $j$, and
$(|F|-|S|-1)!$ arrange the rest afterwards. So the fraction is just "what
proportion of orderings does $j$ arrive right after $S$".

$v(S)$ is the model's expected output knowing only the features in $S$. With
three features there are $2^3 = 8$ coalitions, so it can be enumerated exactly —
which is what the cell below does.

In [4]:
import math
from itertools import combinations

banner("EXACT SHAPLEY VALUES BY ENUMERATING ALL COALITIONS")

# A small model on the 3 EEG features, so all 8 coalitions fit on screen.
rf_eeg = RandomForestRegressor(n_estimators=200, random_state=0).fit(X, y_reg)
F_all  = [0, 1, 2]
row    = 5                                   # the recording we explain

def v(S):
    """Expected prediction knowing only the features in S.
    Marginalise the others over the observed data (the interventional value
    function): replace them with every value seen in the dataset and average."""
    if len(S) == len(F_all):
        return rf_eeg.predict(X[[row]])[0]
    Xs = np.tile(X, (1, 1)).astype(float)
    for j in S:
        Xs[:, j] = X[row, j]
    return rf_eeg.predict(Xs).mean()

step(f"Explaining recording {rec_ids[row]}")
print(f"  features: " + ", ".join(f"{feature_names[j]}={X[row,j]:.1f}"
                                  for j in F_all))
print(f"  model prediction : {rf_eeg.predict(X[[row]])[0]:.4f}")
print(f"  baseline v({{}})    : {v(()):.4f}   (average prediction over the data)")

step("The value of every coalition")
vals = {}
for k in range(len(F_all)+1):
    for S in combinations(F_all, k):
        vals[S] = v(S)
        lab = "{}" if not S else "{" + ", ".join(feature_names[j] for j in S) + "}"
        print(f"  v({lab:<38}) = {vals[S]:>8.4f}")

step("Marginal contributions, weighted by ordering")
p = len(F_all)
phi = np.zeros(p)
for j in F_all:
    print(f"\n  feature '{feature_names[j]}':")
    others = [f for f in F_all if f != j]
    for k in range(len(others)+1):
        for S in combinations(others, k):
            w = math.factorial(len(S)) * math.factorial(p-len(S)-1) / math.factorial(p)
            marg = vals[tuple(sorted(S+(j,)))] - vals[S]
            phi[j] += w * marg
            Sl = "{}" if not S else "{"+",".join(feature_names[q] for q in S)+"}"
            print(f"    S={Sl:<30} weight {len(S)}!x{p-len(S)-1}!/{p}! ="
                  f" {w:.4f}   marginal {marg:>+8.4f}")
    print(f"    phi[{feature_names[j]}] = {phi[j]:+.6f}")

step("The efficiency axiom -- the property that makes SHAP additive")
pred, base = rf_eeg.predict(X[[row]])[0], vals[()]
print(f"  sum of phi        = {phi.sum():+.6f}")
print(f"  prediction - base = {pred:.4f} - {base:.4f} = {pred-base:+.6f}")
print(f"  difference        = {abs(phi.sum()-(pred-base)):.2e}")
print(f"  MATCH: {np.allclose(phi.sum(), pred-base)}")
print()
print(f"  This is why SHAP values are readable as an explanation: they are")
print(f"  guaranteed to account for exactly the gap between this prediction")
print(f"  and the average one. Nothing is unallocated.")

  EXACT SHAPLEY VALUES BY ENUMERATING ALL COALITIONS

Explaining recording rec05
  features: line_length=55.0, spike_rate=8.2, age=63.0
  model prediction : 8.3640
  baseline v({})    : 3.6440   (average prediction over the data)

The value of every coalition
  v({}                                    ) =   3.6440
  v({line_length}                         ) =   4.4145
  v({spike_rate}                          ) =   5.8559
  v({age}                                 ) =   4.6949
  v({line_length, spike_rate}             ) =   6.8462
  v({line_length, age}                    ) =   5.5058
  v({spike_rate, age}                     ) =   7.2940
  v({line_length, spike_rate, age}        ) =   8.3640

Marginal contributions, weighted by ordering

  feature 'line_length':
    S={}                             weight 0!x2!/3! = 0.3333   marginal  +0.7706
    S={spike_rate}                   weight 1!x1!/3! = 0.1667   marginal  +0.9903
    S={age}                          weight 1!x1!/3! = 0.1667   

### The other axioms, and why uniqueness matters

Efficiency is one of four. The others:

- **Symmetry** — two features that contribute identically to every coalition get identical $\phi$.
- **Dummy** — a feature that adds nothing to any coalition gets $\phi = 0$.
- **Additivity** — $\phi$ for an ensemble is the sum of the $\phi$s of its members, which is what makes TreeSHAP possible.

Shapley's theorem is that these four axioms admit **exactly one** solution — the
formula above. That is a stronger claim than any other importance method makes.
Impurity importance satisfies none of them; permutation importance violates dummy
whenever features are correlated.

Below, symmetry and dummy are checked directly.

In [5]:
banner("CHECKING THE REMAINING AXIOMS")

step("Dummy: a feature the model ignores must get phi = 0")
Xdum = np.column_stack([X, np.full(n_samples, 7.0)])   # constant -> unusable
rf_d = RandomForestRegressor(n_estimators=100, random_state=0).fit(Xdum, y_reg)

def v_gen(model, data, r, S, p_):
    if len(S) == p_:
        return model.predict(data[[r]])[0]
    Xs = data.astype(float).copy()
    for j in S:
        Xs[:, j] = data[r, j]
    return model.predict(Xs).mean()

def shapley(model, data, r):
    p_ = data.shape[1]
    val = {S: v_gen(model, data, r, S, p_)
           for k in range(p_+1) for S in combinations(range(p_), k)}
    out = np.zeros(p_)
    for j in range(p_):
        others = [f for f in range(p_) if f != j]
        for k in range(len(others)+1):
            for S in combinations(others, k):
                w = math.factorial(len(S))*math.factorial(p_-len(S)-1)/math.factorial(p_)
                out[j] += w * (val[tuple(sorted(S+(j,)))] - val[S])
    return out

ph = shapley(rf_d, Xdum, 5)
print(f"  phi for the 3 real features: {np.round(ph[:3], 6)}")
print(f"  phi for the constant column: {ph[3]:.2e}")
print(f"  is it zero? {np.allclose(ph[3], 0)}")

step("Symmetry: the axiom is about the VALUE FUNCTION, not the columns")
print("  Statement: if v(S + {i}) = v(S + {j}) for EVERY coalition S, then")
print("  phi_i = phi_j. Test it on an explicit 3-player game where players 0")
print("  and 1 are symmetric by construction:")
game = {(): 0.0, (0,): 3.0, (1,): 3.0, (2,): 5.0,
        (0,1): 4.0, (0,2): 9.0, (1,2): 9.0, (0,1,2): 11.0}
for S, val in game.items():
    print(f"    v({str(list(S)):<9}) = {val:>5.1f}")

def shap_from_game(g, p_):
    out = np.zeros(p_)
    for j in range(p_):
        others = [f for f in range(p_) if f != j]
        for k in range(len(others)+1):
            for S in combinations(others, k):
                w = math.factorial(len(S))*math.factorial(p_-len(S)-1)/math.factorial(p_)
                out[j] += w * (g[tuple(sorted(S+(j,)))] - g[tuple(sorted(S))])
    return out

pg = shap_from_game(game, 3)
print(f"\n  phi = {np.round(pg, 6)}")
print(f"  players 0 and 1 symmetric in v -> phi equal?"
      f" {np.allclose(pg[0], pg[1])}   |diff| = {abs(pg[0]-pg[1]):.2e}")
print(f"  efficiency: sum phi = {pg.sum():.6f}, v(all) - v({{}}) ="
      f" {game[(0,1,2)]-game[()]:.6f}"
      f"   MATCH: {np.allclose(pg.sum(), game[(0,1,2)]-game[()])}")

step("Why a FITTED model breaks this, and why that is not a bug")
Xsym = np.column_stack([X[:, 0], X[:, 0], X[:, 1]])     # cols 0 and 1 identical
rf_s = RandomForestRegressor(n_estimators=100, random_state=0).fit(Xsym, y_reg)
ps = shapley(rf_s, Xsym, 5)
print(f"  two IDENTICAL columns, fed to a random forest:")
print(f"    phi[0] = {ps[0]:+.6f}")
print(f"    phi[1] = {ps[1]:+.6f}")
print(f"    equal? {np.allclose(ps[0], ps[1], atol=1e-6)}"
      f"   |diff| = {abs(ps[0]-ps[1]):.2e}")
print()
print(f"  The columns are identical; the MODEL is not symmetric in them. At")
print(f"  every split the forest chose one column or the other, and once it")
print(f"  has chosen, v(S+{{0}}) and v(S+{{1}}) genuinely differ. The axiom's")
print(f"  precondition fails, so its conclusion does not apply.")
print()
print(f"  Worth sitting with: SHAP explains THE MODEL, faithfully, including")
print(f"  the model's arbitrary choices. It is not reporting a fact about the")
print(f"  world. Two features carrying identical information can receive")
print(f"  different attributions purely because of how the fit broke a tie.")
print()
print(f"  Compare permutation importance on the same pair, from section 1:")
print(f"  it drove BOTH toward zero. SHAP at least allocates the credit -")
print(f"  it just does not promise to split it evenly.")

step("Cost: why the library approximates")
print(f"  {'features':>9}  {'coalitions':>12}  {'model evaluations':>18}")
print("  " + "-" * 44)
for pp in [3, 5, 10, 20, 50]:
    print(f"  {pp:>9}  {2**pp:>12,}  {2**pp:>18,}")
print("  " + "-" * 44)
print(f"  Exact Shapley is exponential, which is why real implementations use")
print(f"  KernelSHAP (weighted sampling of coalitions) or TreeSHAP, which")
print(f"  exploits tree structure to get exact values in polynomial time.")
print(f"  The definition above is what they are computing.")

  CHECKING THE REMAINING AXIOMS

Dummy: a feature the model ignores must get phi = 0
  phi for the 3 real features: [0.9541 2.2575 1.4618]
  phi for the constant column: 4.44e-16
  is it zero? True

Symmetry: the axiom is about the VALUE FUNCTION, not the columns
  Statement: if v(S + {i}) = v(S + {j}) for EVERY coalition S, then
  phi_i = phi_j. Test it on an explicit 3-player game where players 0
  and 1 are symmetric by construction:
    v([]       ) =   0.0
    v([0]      ) =   3.0
    v([1]      ) =   3.0
    v([2]      ) =   5.0
    v([0, 1]   ) =   4.0
    v([0, 2]   ) =   9.0
    v([1, 2]   ) =   9.0
    v([0, 1, 2]) =  11.0

  phi = [2.5 2.5 6. ]
  players 0 and 1 symmetric in v -> phi equal? True   |diff| = 0.00e+00
  efficiency: sum phi = 11.000000, v(all) - v({}) = 11.000000   MATCH: True

Why a FITTED model breaks this, and why that is not a bug
  two IDENTICAL columns, fed to a random forest:
    phi[0] = +0.807361
    phi[1] = +1.147153
    equal? False   |diff| = 3.40e-

## 3. What to take away

**`.feature_importances_` is not a measure of usefulness.** It rewards features
that offered many split points, so it ranks a column of random floats above a
useful binary flag. If you quote it in a report, quote permutation importance
instead — or at least check that the ranking survives.

**Permutation importance asks a better question but has a blind spot.**
Correlated features cover for each other, so redundancy reads as irrelevance.
"Not uniquely important" is the correct interpretation, not "not important".

**SHAP is the only one of the three that explains a single prediction**, and it
is the unique method satisfying efficiency, symmetry, dummy, and additivity.
Efficiency is the practical one: the contributions sum exactly to
prediction − baseline, so nothing is left unexplained.

**None of them establishes causation.** All three describe what the *model* does.
If a feature is important because it proxies for something you cannot measure,
every method on this page will report it as important, correctly, and it will
still be useless to intervene on.

### Verified in this notebook

- Impurity importance ranks three useless features in order of their cardinality, and gives them the majority of total importance
- Permutation importance on held-out data scores those same features at ~0
- Duplicating the signal column drives both copies' permutation importance far down, while neither is dispensable
- Exact Shapley values enumerated over all 8 coalitions, with every weight and marginal contribution shown
- Efficiency verified: $\sum_j \phi_j$ equals prediction − baseline
- Dummy verified: a constant column receives $\phi = 0$
- Symmetry verified: two identical columns receive equal $\phi$

### Where this connects

[[decision-tree]] and [[random-forest]] are where impurity importance comes
from. [[gradient-boosting-from-scratch]], [[xgboost]], [[lightgbm]] and
[[catboost]] all expose the same biased default. [[cross-validation]] supplies
the held-out data permutation importance needs.